# Лабораторная 1: декодерный трансформер для предсказания следующего токена

## Что нужно знать до старта
Перед началом лабораторной полезно открыть:
- [../README.md](./README.md)
- [guides/00_autoregression_prerequisites.md](./guides/00_autoregression_prerequisites.md)
- [guides/01_decoder_only_toy_walkthrough.md](./guides/01_decoder_only_toy_walkthrough.md)
- [../theory/theory.md](../theory/theory.md)

Это первая лабораторная блока `04-Autoregression` и Шаг 7 общего курса.


## Выбор среды выполнения

Рекомендуемый стартовый режим: `auto`.

Если нужна полностью воспроизводимая проверка на центральном процессоре, выберите `local-cpu` и выполните `Restart & Run All`.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/04-Autoregression/lab/requirements.txt"


def _detect_notebook_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


## Интуиция задачи без формул

Модель предсказывает следующий токен, опираясь только на прошлые позиции. Для этого в блоке внимания используется причинная маска (causal mask).


## Как проходить работу

Маршрут строго фиксирован: данные → маска → блок декодера → обучение → генерация → диагностика внимания.


## Контракт данных

В этой лабораторной используются последовательности фиксированной максимальной длины.

- Вход `X`: идентификаторы токенов без последнего шага, форма `(N, T)`.
- Цель `Y`: те же последовательности, сдвинутые на один шаг влево, форма `(N, T)`.
- Дополнение пустыми позициями задаётся токеном `PAD = 0`.

Разбиение на обучение, проверку и тест выполняется строго индексами без перемешивания.


## Таблица ключевых форм

| Сущность | Смысл | Форма |
|---|---|---|
| `X` | входные токены | `(N, T)` |
| `Y` | цели следующего токена | `(N, T)` |
| `mask` | причинная маска | `(T, T)` |
| `attention_scores` | веса внимания | `(N, H, T, T)` |
| `y_pred` | вероятности токенов | `(N, T, V)` |


## Ручной пример

Пусть последовательность имеет вид:

```text
<BOS> A B C <EOS> <PAD> <PAD>
```

Тогда пара для обучения следующему токену строится так:

```text
X: <BOS> A B C <EOS> <PAD>
Y: A B C <EOS> <PAD> <PAD>
```

То есть на каждом шаге модель должна восстановить токен, стоящий на одну позицию правее.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 13
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
TOKEN_A = 3
TOKEN_B = 4
TOKEN_C = 5
TOKEN_D = 6

ID_TO_TOKEN = {
    PAD_ID: '<PAD>',
    BOS_ID: '<BOS>',
    EOS_ID: '<EOS>',
    TOKEN_A: 'A',
    TOKEN_B: 'B',
    TOKEN_C: 'C',
    TOKEN_D: 'D',
}
VOCAB_SIZE = len(ID_TO_TOKEN)
MAX_SEQ_LEN = 18
TRAIN_SAMPLES = 4096
VAL_SAMPLES = 512
TEST_SAMPLES = 512
TOTAL_SAMPLES = TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES

EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
BATCH_SIZE = 64
EPOCHS = 8

plt.style.use('default')
keras.utils.set_random_seed(SEED)


## TODO 1: подготовка детерминированных данных


In [ ]:
def build_synthetic_corpus(num_samples, max_seq_len=MAX_SEQ_LEN, seed=SEED):
    """Создаёт детерминированный синтетический корпус для лабораторной.

    Args:
      num_samples: Число последовательностей в корпусе.
      max_seq_len: Максимальная длина последовательности с учётом дополнения.
      seed: Зерно случайности для воспроизводимости.

    Returns:
      Кортеж `(X, Y)`, где обе матрицы имеют форму `(num_samples, max_seq_len - 1)`.

    Raises:
      ValueError: Если `max_seq_len` слишком мал для корректной сборки последовательностей.
    """
    # TODO 1.1: Реализуйте генерацию детерминированных последовательностей.
    # Подсказка: используйте фиксированное правило перехода между токенами.
    raise NotImplementedError('TODO 1: реализуйте build_synthetic_corpus')


# TODO 1.2: Постройте X_all, y_all и выполните фиксированное индексное разбиение
# на X_train/y_train, X_val/y_val, X_test/y_test без перемешивания.
raise NotImplementedError('TODO 1: подготовьте детерминированное разбиение')


In [ ]:
# Мини-проверка после TODO 1
print('После реализации TODO 1 здесь должны выводиться формы train/val/test.')


## TODO 2: причинная маска и её проверка


In [ ]:
def build_causal_mask(seq_len):
    """Строит нижнетреугольную причинную маску.

    Args:
      seq_len: Длина последовательности.

    Returns:
      Булев тензор формы `(seq_len, seq_len)`.

    Raises:
      ValueError: Если `seq_len` не является положительным числом.
    """
    # TODO 2.1: Верните нижнетреугольную причинную маску.
    raise NotImplementedError('TODO 2: реализуйте build_causal_mask')


def assert_lower_triangular(mask):
    """Проверяет, что маска имеет нижнетреугольную структуру.

    Args:
      mask: Булева матрица формы `(T, T)`.

    Raises:
      AssertionError: Если верхний треугольник содержит разрешающие значения.
    """
    array_mask = np.asarray(mask).astype(np.int32)
    upper = np.triu(array_mask, k=1)
    assert upper.sum() == 0, 'Верхний треугольник должен быть полностью закрыт.'


# TODO 2.2: Постройте пример маски длины 8, проверьте форму и нижнетреугольность.
raise NotImplementedError('TODO 2: проверьте причинную маску')


## TODO 3: декодерный блок и метрики


In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    """Складывает векторы токенов и позиций в едином представлении.

    Args:
      maxlen: Максимальная длина последовательности.
      vocab_size: Размер словаря.
      embed_dim: Размер скрытого представления.
    """

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        """Инициализирует таблицы токенных и позиционных векторов.

        Args:
          maxlen: Максимальная длина последовательности.
          vocab_size: Размер словаря.
          embed_dim: Размер векторного представления.
          **kwargs: Дополнительные аргументы базового класса.
        """
        super().__init__(**kwargs)
        self.maxlen = maxlen
        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        """Вычисляет сумму токенных и позиционных векторов."""
        # TODO 3.1: Постройте позиции через tf.range и сложите токенные/позиционные векторы.
        raise NotImplementedError('TODO 3: реализуйте TokenAndPositionEmbedding.call')

    def compute_mask(self, inputs, mask=None):
        """Переадресует маску непустых токенов из слоя токенных векторов."""
        return self.token_embedding.compute_mask(inputs)


class CausalDecoderBlock(layers.Layer):
    """Минимальный декодерный блок с причинной маской."""

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        """Создаёт внутренние слои блока."""
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')

        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ],
            name='позиционно_независимая_сеть',
        )
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(rate)
        self.dropout_2 = layers.Dropout(rate)

    def call(self, inputs, padding_mask=None, training=None, return_attention_scores=False):
        """Прогоняет вход через внимание и позиционно-независимую сеть."""
        # TODO 3.2: Соберите combined mask = causal mask + padding mask.
        # TODO 3.3: Выполните self-attention, residual, normalization, ffn, residual, normalization.
        raise NotImplementedError('TODO 3: реализуйте CausalDecoderBlock.call')

    def compute_mask(self, inputs, mask=None):
        """Передаёт маску дальше без изменений."""
        return mask


def masked_sparse_crossentropy(y_true, y_pred):
    """Считает перекрёстную энтропию только по непустым позициям.

    Args:
      y_true: Истинные идентификаторы токенов.
      y_pred: Предсказанные вероятности токенов.

    Returns:
      Среднее значение функции потерь по непустым позициям.
    """
    per_token = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    masked = per_token * mask
    return tf.reduce_sum(masked) / tf.reduce_sum(mask)


def masked_token_accuracy(y_true, y_pred):
    """Считает токенную точность только по непустым позициям.

    Args:
      y_true: Истинные идентификаторы токенов.
      y_pred: Предсказанные вероятности токенов.

    Returns:
      Доля верных предсказаний по непустым позициям.
    """
    predicted = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    correct = tf.cast(tf.equal(y_true, predicted), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(correct * mask) / tf.reduce_sum(mask)


def perplexity_from_loss(loss_value):
    """Преобразует среднюю перекрёстную энтропию в перплексию.

    Args:
      loss_value: Среднее значение функции потерь.

    Returns:
      Значение перплексии.
    """
    return float(np.exp(loss_value))


## TODO 4: сборка и обучение модели


In [ ]:
# TODO 4.1: Соберите модель с TokenAndPositionEmbedding и CausalDecoderBlock.
# TODO 4.2: Скомпилируйте модель с masked_sparse_crossentropy и masked_token_accuracy.
# TODO 4.3: Обучите модель на train и validation без перемешивания.
raise NotImplementedError('TODO 4: соберите и обучите модель')


In [ ]:
# Графики после TODO 4
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='обучение')
plt.plot(history.history['val_loss'], label='проверка')
plt.title('Перекрёстная энтропия по эпохам')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
train_ppl = [perplexity_from_loss(v) for v in history.history['loss']]
val_ppl = [perplexity_from_loss(v) for v in history.history['val_loss']]
plt.plot(train_ppl, label='обучение')
plt.plot(val_ppl, label='проверка')
plt.title('Перплексия по эпохам')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()


In [ ]:
# Контроль порогов после TODO 4
test_loss, test_token_acc = model.evaluate(X_test, y_test, verbose=0)
test_perplexity = perplexity_from_loss(test_loss)
print(f'test_loss            : {test_loss:.4f}')
print(f'test_token_accuracy  : {test_token_acc:.4f}')
print(f'test_perplexity      : {test_perplexity:.4f}')
assert test_token_acc >= 0.97, 'Токенная точность ниже порога 0.97.'
assert test_perplexity <= 1.30, 'Перплексия выше порога 1.30.'


## TODO 5: детерминированная генерация


In [ ]:
def generate_greedy(model, prompt_ids, max_new_tokens, max_seq_len=MAX_SEQ_LEN - 1):
    """Генерирует продолжение в детерминированном режиме argmax.

    Args:
      model: Обученная модель предсказания следующего токена.
      prompt_ids: Начальная последовательность идентификаторов токенов.
      max_new_tokens: Максимум новых шагов генерации.
      max_seq_len: Рабочая длина входа модели.

    Returns:
      Список идентификаторов с учётом исходной подсказки и сгенерированных токенов.

    Raises:
      ValueError: Если подсказка пуста.
    """
    # TODO 5.1: Реализуйте детерминированную генерацию через argmax.
    raise NotImplementedError('TODO 5: реализуйте generate_greedy')


# TODO 5.2: Запустите 20 фиксированных подсказок и проверьте порог 18/20.
raise NotImplementedError('TODO 5: проверьте качество генерации')


## TODO 6: диагностика внимания


In [ ]:
# TODO 6.1: Соберите tracing-path, который возвращает attention_scores из CausalDecoderBlock.
# TODO 6.2: Усредните attention_scores по головам и обрежьте карту до непустых токенов.
# TODO 6.3: Проверьте, что отношение массы внимания в будущем меньше 1e-4.
raise NotImplementedError('TODO 6: выполните диагностику внимания')


## Чек-лист перед завершением
1. Все `TODO` закрыты.
2. Воспроизводимость данных подтверждена на первых `k` примерах.
3. Токенная точность и перплексия проходят пороги.
4. Детерминированная генерация выполняет порог `18/20`.
5. Карта внимания не содержит доступа к будущим позициям.
